# The SSR under quadratic rough Heston

We compute the SSR by blipping the forward volatility curve. 

The resulting term structures cover the four reference dates from the paper.

In [ ]:
if (!file.exists("paper_figures.R")) setwd("R")
source("paper_figures.R")
OUTPUT <- "../figs/reproduction/R"
dir.create(OUTPUT, recursive = TRUE, showWarnings = FALSE)

## Gamma kernel and initial forward variance curve

We use the gamma kernel
$$\kappa(\tau)=\frac{\nu}{\Gamma(\alpha)}\,\tau^{\alpha-1}e^{-\lambda\tau},\qquad \alpha=H+\tfrac12,$$
and construct $\xi_0$ from SPX variance swaps with smoothing `EPS = 0.03`. 

We obtain $y_0$ from
$$y_0(u)^2=\xi_0(u)-c-\int_0^u\xi_0(s)\,\kappa(u-s)^2\,ds,$$
using Gauss–Jacobi quadrature.

## Blipped forward volatility curves

In forward volatility form, the model reads $dy_t(u)=\kappa(u-t)\sqrt{V_t}\,dW_t$. Since the spot-volatility correlation is $-1$, the finite-difference methodology of Bourgey et al. uses
$$y_t^h(u)=y_t(u)-h\,\kappa(u-t),\qquad
\beta_t(T)\approx\frac{\sigma_t(T,y_t^h)-\sigma_t(T,y_t)}{h}.$$

Experiment shows that scaling the blip by time to expiration works well. 

Here $h(T)=\texttt{BLIP}\sqrt{T}$ with `BLIP = 0.01`, as written in the paper.

Base and blipped simulations share the same random numbers.

In [ ]:
SAVEFIG <- FALSE  # FALSE: display only; TRUE: export PDF/EPS and CSV.
PATHS <- 300000; STEPS <- 300; SEED <- 1234; EPS <- .03; BLIP <- 0.01

## Compute base and blipped implied volatilities and the ATM skew

We estimate the regression coefficient from the difference in ATM implied volatilities. The ATM-forward skew is
$$\mathcal{S}_0(T)=\frac{\Phi(-\sqrt{w}/2)-\mathbb{P}(S_T\geq F_T)}{\sqrt{T}\,\phi(-\sqrt{w}/2)},
\qquad w=T\sigma_{\mathrm{ATM}}^2,$$
where $F_T$ is the sample forward and $\Phi,\phi$ are the standard normal CDF and density. The SSR is $\mathcal{R}_0(T)\approx\beta_0(T)/\mathcal{S}_0(T)$.

In [ ]:
ssr <- paper.ssr(PATHS, STEPS, seed = SEED, eps = EPS, h = BLIP)

## Plot the SSR term structures

We use all maturities through one year and add the analytical limit $H+3/2$ at zero.

In [ ]:
paper.plot.ssr(ssr)
if (SAVEFIG) {
  paper.save(function() paper.plot.ssr(ssr), OUTPUT, "ssr4_ts")
  write.csv(ssr, file.path(OUTPUT, "ssr4_ts.csv"), row.names = FALSE)
}